In [10]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import numpy as np
import pandas as pd
import time
import scipy.stats as spst
import scipy.special as spsp
from fetch_deribit_data import *
from callibration_32model import *

import sys
sys.path.insert(sys.path.index("")+1, "C:/Users/27261/Desktop/3_Courses in PHBS/3_09_AppliedStochasticProcess/Project_sv32_EMC")
import pyfeng as pf
import pyfeng.ex as pfex
from utils import *

In [ ]:
if __name__ == "__main__":
    # 放宽窗口时间，收集充足的散点
    periods = {
        "1_Pre_Shock":  ("2025-10-10 20:55:00", "2025-10-10 21:00:00"), # 5分钟
        "2_Crash":      ("2025-10-10 21:15:00", "2025-10-10 21:20:00"), # 5分钟
        "3_Recovery":   ("2025-10-10 23:05:00", "2025-10-10 23:10:00")  # 5分钟
    }
    
    for phase_name, (start_t, end_t) in periods.items():
        print(f"\n====== 正在处理阶段: {phase_name} ({start_t} 到 {end_t}) ======")
        raw_df = fetch_deribit_trades_robust(start_t, end_t)
        
        if not raw_df.empty:
            # 1. 保存未过滤的原始数据 (Raw Data)
            raw_csv_filename = f"Deribit_RAW_{phase_name}.csv"
            raw_df.to_csv(raw_csv_filename, index=False)
            print(f"[成功] 阶段 {phase_name} 的原始成交记录 ({len(raw_df)} 条) 已保存至 {raw_csv_filename}")
            
            # 2. 清洗并保存过滤后的 OTM 数据
            print(f"[处理] 正在对 {phase_name} 进行动态 Moneyness 清洗...")
            clean_otm_df = process_and_filter_otm_dynamic(raw_df)
            
            otm_csv_filename = f"Deribit_OTM_Moneyness_{phase_name}.csv"
            clean_otm_df.to_csv(otm_csv_filename, index=False)
            print(f"[成功] 得到可用 OTM 样本 {len(clean_otm_df)} 个，已保存至 {otm_csv_filename}！")
        else:
            print(f"[警告] 阶段 {phase_name} 未拉取到数据！")


====== 正在处理阶段: 1_Pre_Shock (2025-10-10 20:55:00 到 2025-10-10 21:00:00) ======
[成功] 阶段 1_Pre_Shock 的原始成交记录 (469361 条) 已保存至 Deribit_RAW_1_Pre_Shock.csv
[处理] 正在对 1_Pre_Shock 进行动态 Moneyness 清洗...
[成功] 得到可用 OTM 样本 6 个，已保存至 Deribit_OTM_Moneyness_1_Pre_Shock.csv！

====== 正在处理阶段: 2_Crash (2025-10-10 21:15:00 到 2025-10-10 21:20:00) ======
[成功] 阶段 2_Crash 的原始成交记录 (413088 条) 已保存至 Deribit_RAW_2_Crash.csv
[处理] 正在对 2_Crash 进行动态 Moneyness 清洗...
[成功] 得到可用 OTM 样本 19 个，已保存至 Deribit_OTM_Moneyness_2_Crash.csv！

====== 正在处理阶段: 3_Recovery (2025-10-10 23:05:00 到 2025-10-10 23:10:00) ======
  [网络波动] 抓取失败 (Response ended prematurely). 第 1/5 次重试中...
  [网络波动] 抓取失败 (HTTPSConnectionPool(host='history.deribit.com', port=443): Max retries exceeded with url: /api/v2/public/get_last_trades_by_currency_and_time?currency=BTC&kind=option&start_timestamp=1760137766030&end_timestamp=1760137800000&count=1000 (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.

In [9]:
if __name__ == "__main__":
    files_to_calibrate = [
        "Deribit_OTM_Moneyness_1_Pre_Shock_LONG_TERM.csv",
    ]
    
    results_dict = {}
    
    for file in files_to_calibrate:
        params = run_calibration(file, r=0.1)
        if params is not None:
            results_dict[file] = params


[Deribit_OTM_Moneyness_1_Pre_Shock_LONG_TERM.csv] 开始加载数据并进行 3/2 模型校准...
期权数量: 20, 到期时间 T: 0.20948, 初始猜测 V0: 0.1818
优化器启动中，请耐心等待 (约需 1~3 分钟)...
✅ 校准成功！耗时: 3.8 秒
------------------------------
V0    (瞬时方差) = 0.2405  (相当于瞬时 IV = 49.04%)
kappa (回归速度) = 1.9392
theta (长期方差) = 0.0480
sigma (Vol-of-Vol) = 4.1822
rho   (相关系数) = -0.9990
------------------------------
 Moneyness Type    iv  Market_Norm_Price  Model_Norm_Price  Error_%
  0.349429    P 88.17           0.000282          0.000264    -6.71
  0.615347    P 65.24           0.004241          0.003831    -9.67
  0.697940    P 58.22           0.007541          0.007816     3.64
  0.750844    P 54.68           0.011287          0.011930     5.70
  0.785826    P 52.44           0.014624          0.015690     7.29
  0.838021    P 48.72           0.020718          0.023246    12.20
  0.875089    P 46.55           0.026937          0.030422    12.94
  0.907965    P 44.84           0.034029          0.038281    12.49
  0.917911    P 44.98      